# Step 5b. 개별 리뷰 재수집 (보완 B + C용)

**목표**: 50개 게임의 개별 리뷰를 수집하여 플레이타임·긍정 여부를 포함한 데이터 확보  
**입력**: `data/target_games.csv`  
**출력**: `data/review_individual.csv`  
**API**: Steam `appreviews` (무료, 인증 불필요)

### 기존 05_reviews.ipynb와 차이점
- 일별 집계가 아닌 **개별 리뷰 행** 저장
- `playtime_at_review_min`, `playtime_forever_min`, `voted_up` 포함 → 보완 B 민감도 분석 + 보완 C 긍정률 분석에 사용
- `filter_offtopic_activity=1` 활성화 (review bombing 완화)
- 게임당 최대 50,000개 리뷰 수집 (Rust·Terraria 등 초고인기작 무한루프 방지)
- resume 기능: 중단 후 재실행 시 이미 수집된 게임 자동 스킵

### 예상 소요 시간
- sleep 1.0초 × 페이지당 100개 기준, 게임당 평균 수백~수천 페이지
- **수 시간 단위** — 별도 터미널 또는 백그라운드 실행 권장
- 중단 시 재실행하면 resume 기능으로 이어서 진행

In [ ]:
import requests
import pandas as pd
import time
import os
from datetime import datetime, timezone

target_games = pd.read_csv("../data/target_games.csv")
print(f"수집 대상: {len(target_games)}개 게임")
print()
print("장르별 게임 수:")
print(target_games.groupby("genre_category")["appid"].count().rename("게임 수"))

## 설정값

- `MAX_REVIEWS_PER_GAME`: 게임당 최대 수집 리뷰 수. Terraria(140만), Rust(120만) 등 초고인기작 대비 상한 설정.
- `SLEEP_SEC`: API 호출 간 딜레이. 1.0초 이상 권장 (Steam rate limit 방지).

In [ ]:
import requests
import pandas as pd
import time
import os
from datetime import datetime, timezone, timedelta

SLEEP_SEC            = 1.0
MAX_REVIEWS_PER_GAME = 50_000
MAX_RETRIES          = 3
OUTPUT_PATH          = "../data/review_individual.csv"

SINCE_DT = datetime.now(timezone.utc) - timedelta(days=3 * 365)
SINCE_TS = int(SINCE_DT.timestamp())

print(f"sleep: {SLEEP_SEC}초 / 게임당 최대: {MAX_REVIEWS_PER_GAME:,}개 / 재시도: {MAX_RETRIES}회")
print(f"수집 기간: {SINCE_DT.strftime('%Y-%m-%d')} 이후")
print(f"출력: {OUTPUT_PATH}")

## 단일 게임 수집 함수

- cursor 방식 페이지네이션 (다음 cursor가 이전과 같거나 reviews가 빈 배열이면 중단)
- HTTP 에러 발생 시 최대 3회 재시도, 지수 백오프
- `filter_offtopic_activity=1`: Steam API의 off-topic review bomb 기본 필터 활성화

In [ ]:
def collect_game_reviews(appid, max_reviews=MAX_REVIEWS_PER_GAME):
    collected = []
    cursor = "*"

    while len(collected) < max_reviews:
        params = {
            "json": 1,
            "filter": "recent",
            "language": "all",
            "review_type": "all",
            "purchase_type": "all",
            "num_per_page": 100,
            "cursor": cursor,
            "filter_offtopic_activity": 1,
        }

        data = None
        for attempt in range(MAX_RETRIES):
            try:
                resp = requests.get(
                    f"https://store.steampowered.com/appreviews/{appid}",
                    params=params,
                    timeout=20,
                )
                resp.raise_for_status()
                data = resp.json()
                break
            except Exception as e:
                if attempt == MAX_RETRIES - 1:
                    print(f"  ⚠️ 재시도 초과 (appid={appid}): {e}")
                    return collected
                wait = 2 ** (attempt + 1)
                print(f"  재시도 {attempt + 1}/{MAX_RETRIES} ({wait}초 대기)...", flush=True)
                time.sleep(wait)

        reviews = data.get("reviews", [])
        if not reviews:
            break

        new_cursor = data.get("cursor", "")
        if not new_cursor or new_cursor == cursor:
            break
        cursor = new_cursor

        hit_cutoff = False
        for r in reviews:
            if r.get("timestamp_created", 0) < SINCE_TS:
                hit_cutoff = True
                continue  # cutoff 이전 리뷰는 수집하지 않음

            author = r.get("author", {})
            text   = r.get("review", "")
            collected.append({
                "review_id"             : r.get("recommendationid"),
                "timestamp_created"     : r.get("timestamp_created"),
                "voted_up"              : r.get("voted_up"),
                "playtime_at_review_min": author.get("playtime_at_review"),
                "playtime_forever_min"  : author.get("playtime_forever"),
                "language"              : r.get("language"),
                "review_length"         : len(text) if text else 0,
            })

        time.sleep(SLEEP_SEC)

        if hit_cutoff:
            break  # 이 페이지에서 cutoff 이전 리뷰 발견 → 중단

    return collected


print("함수 정의 완료")

## 전체 게임 루프 (resume 기능 포함)

- 출력 파일이 이미 있으면 수집된 `app_id` 목록을 읽어 자동 스킵
- 한 게임 완료 시마다 CSV에 append → 중단 시 데이터 손실 없음

In [ ]:
# Resume: 이미 수집된 app_id 확인
if os.path.exists(OUTPUT_PATH):
    existing = pd.read_csv(OUTPUT_PATH, usecols=["app_id"])
    done_appids = set(existing["app_id"].unique())
    print(f"Resume 모드: {len(done_appids)}개 게임 이미 수집됨 → 스킵")
else:
    done_appids = set()
    print("신규 수집 시작")

total = len(target_games)

for i, (_, game) in enumerate(target_games.iterrows(), start=1):
    appid = int(game["appid"])
    name  = game["name"]
    genre = game["genre_category"]

    if appid in done_appids:
        print(f"[{i:02d}/{total}] {name} — 스킵")
        continue

    print(f"[{i:02d}/{total}] {name} ({appid}) 수집 중...", flush=True)
    start_time = time.time()

    reviews = collect_game_reviews(appid)

    if reviews:
        df_game = pd.DataFrame(reviews)
        df_game.insert(0, "app_id",   appid)
        df_game.insert(1, "app_name", name)
        df_game.insert(2, "genre",    genre)

        write_header = not os.path.exists(OUTPUT_PATH)
        df_game.to_csv(OUTPUT_PATH, mode="a", index=False,
                       header=write_header, encoding="utf-8-sig")

        elapsed = time.time() - start_time
        print(f"  → {len(reviews):,}개 저장 ({elapsed:.0f}초)")
    else:
        print(f"  → 리뷰 없음 또는 수집 실패")

print()
print("=" * 40)
print("전체 수집 완료")

## 수집 결과 요약

수집이 완료된 후 실행하여 데이터 품질 확인.

In [ ]:
result = pd.read_csv(OUTPUT_PATH)

print(f"총 리뷰 수  : {len(result):,}개")
print(f"게임 수     : {result['app_id'].nunique()}개")
print()

print("=== 게임별 수집 리뷰 수 (장르 포함) ===")
per_game = (
    result.groupby(["app_name", "genre"])["review_id"]
    .count()
    .rename("리뷰 수")
    .sort_values(ascending=False)
)
print(per_game.to_string())
print()

print("=== 결측치 확인 ===")
missing = result[["playtime_at_review_min", "playtime_forever_min", "voted_up", "language"]].isna().sum()
missing_pct = (missing / len(result) * 100).round(1)
print(pd.DataFrame({"결측 수": missing, "결측률(%)": missing_pct}))
print()

print("=== voted_up 분포 ===")
print(result["voted_up"].value_counts().rename(index={True: "긍정", False: "부정"}))
print()

print("=== playtime_at_review_min 기초 통계 ===")
print(result["playtime_at_review_min"].describe().round(1))